In [9]:
import os
from datetime import datetime
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from dotenv import load_dotenv

load_dotenv()
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    groq_api_key=os.environ.get("GROQ_API_KEY")
)
tavily = TavilySearch(
    max_result=5,
    tavily_api_key=os.environ.get("TAVILY_API_KEY")
)
@tool
def calculate(a:float,b:float, operation_type:str ):
    """Performs basic arithmetic using two numbers and one operation. Supported operations are
            addition (+), subtraction (-), multiplication (*), and division (/). Division by zero is not allowed.l"""
    operation_type = operation_type.strip().lower()
    if operation_type == "+":
        return a+b
    elif operation_type == "-":
        return a-b
    elif operation_type == "*":
        return a*b
    elif operation_type == "/" and b != 0:
        return a/b
    else:
        raise ValueError(f"Unsupported operation: {operation_type}")
@tool
def get_time():
    """Get the current date and time."""
    return datetime.now().isoformat()
@tool
def web_search(query:str):
    """Search the web for current information."""
    result = tavily.invoke({
        "query": query
    })
    return str(result)
def streamIt(response):
    full_response = ""
    for chunk in response:
        print(chunk.content, end="", flush=True)
        full_response += chunk.content
    return full_response
agent = create_agent(
    model=llm,
    tools=[calculate, get_time, web_search],
    system_prompt=("""You are a tutor for school-going kids. Your answers should be short and easy to understand.""")
)
while True:
    user_input = input("Enter your question...").strip()
    if user_input.lower() in ["stop", "exit", "break", "quit"]:
        break
    response = agent.invoke({
        "messages":[
            {
                "role":"user",
                "content": user_input
            }
        ]
    })
    final_message = response["messages"][-1]
    print("AI:", final_message.content)

AI: Dividing any number by zero (like 20 ÷ 0) doesn’t give a real answer—it’s **undefined**. In math, you can’t divide by zero because there’s no number that you can multiply by 0 to get 20. So the expression “20 divided by zero” has no value.
AI: Dividing something by “none” (zero) can’t be done—​the result would be undefined. So you can’t split 10 bananas into 0 groups. Instead, you could share them into 1, 2, 5, 10, etc., groups.
AI: Dividing 10 bananas by the number of people who have landed on the Sun isn’t possible—no one has ever landed on the Sun, so that number is 0. Division by zero is undefined, so the calculation can’t be done.
